# nano Beatrix v1.1 — the AutoModel release of every folder

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_1_release.ipynb)

The twins repo (`AbstractPhil/alephllm-nano-castor-pollux`) holds the four v1.1 bodies and, beside each, its continuation arms (frames, control, carry, frames-then-carry, notes). Until 0.1.43 only the four bodies were AutoModel packages; the arm folders were training records alone. This notebook packages EVERY craft folder the repo holds through the same gates (the vendored code equals the library's, the weights load strictly, the logits match exactly, a load in a clean child by both routes), writes `crafts.json` (the index of every packaged folder: body, arm, source, steps, parameters, the end reads) and the card from the same facts, pushes it all in one commit, then loads every folder from the repo again.

| cell | what it does | time |
|---|---|---|
| C1 | the pinned install, the token, the decision block | a minute |
| C2 | preflight: the repo's craft folders, which are packaged already, the writer (nothing written) | seconds |
| C3 | the release: fetch each folder's final files, package, gate, index, card, one commit, the load gate from the repo | a few minutes a folder on the CPU (the models are 9M parameters); about an hour for 18 folders |

Needs: an `HF_TOKEN` secret with write access to the repo. A GPU is not needed. The run is idempotent: a folder already packaged is packaged again from the same files and the bytes are identical.

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
%pip install -q "geolip-nano-beatrix-trainer @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.45"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import nano_v1 as V1, automodel as AM
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__)
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 45), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
assert HF_TOKEN, 'HF_TOKEN missing: the packages and the card are pushed to the repo'
os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK ============
OUT = '/content/nano_v1_1_release'                   # the session folder: the fetched finals, the packages, the gates
REPO = V1.REPO                                        # AbstractPhil/alephllm-nano-castor-pollux
CRAFTS = None                                         # None: every craft folder the repo holds; or a list, e.g. ['nano-pollux-dual-v1.1-cf-nt']
print('decision block: repo', REPO, '| crafts', 'every folder in the repo' if CRAFTS is None else CRAFTS, '| session folder', OUT)

In [ ]:
# C2 (P) — PREFLIGHT: the repo's craft folders (read without the token), which hold a package already, and the writer. Nothing is written.
from huggingface_hub import HfApi
api = HfApi(token=False)
paths = [f.path for f in api.list_repo_tree(REPO, recursive=True)]
folders = AM.repo_crafts(REPO, token=False)
rows = []
for n in folders:
    packaged = all(f'{n}/{f}' in paths for f in AM.PACKAGE)
    rows.append((n, V1.body_of(n), V1.arm_text(n) or 'the body', V1.ALL[n].get('source') or '-', 'packaged' if packaged else 'record only'))
print(f'{len(folders)} craft folders in {REPO}:')
for r in rows:
    print('  %-34s body %-28s arm %-20s from %-30s %s' % r)
todo = folders if CRAFTS is None else list(CRAFTS)
assert all(n in folders for n in todo), f'not in the repo: {[n for n in todo if n not in folders]}'
print('to package:', len(todo), 'folders;', sum(1 for r in rows if r[4] == 'record only' and r[0] in todo), 'of them have no package yet')
me = HfApi(token=HF_TOKEN).whoami()
print('writer:', me.get('name'), '| the commit will carry the code at the top, every package, crafts.json and README.md')

In [ ]:
# C3 — THE RELEASE: fetch each folder's final files, package through every gate, write crafts.json and the card, one commit, then the
# load gate again from the repo for every folder. Stop and read the gate file it names if a gate fails: nothing is pushed before every
# gate passed on exactly the bytes pushed.
import json, time
t0 = time.time()
oid = AM.release(OUT, REPO, CRAFTS, HF_TOKEN)
print(f'pushed {oid} to {REPO} in {(time.time() - t0) / 60:.0f} min')
idx = json.load(open(f'{OUT}/automodel/crafts.json', encoding='utf-8'))
print(len(idx), 'folders indexed:')
for n, r in idx.items():
    print('  %-34s %-20s steps %6s -> %6s  params %s' % (n, r['arm'] or 'the body', r['start_step'], r['step'], f"{r['parameters']:,}"))
card = open(f'{OUT}/automodel/README.md', encoding='utf-8').read()
i = card.find('## The continuation arms')
print(card[i:i + 1500] if i >= 0 else 'no arm section (no arm packaged)')

### After

Every folder of the repo now loads with `AutoModelForCausalLM.from_pretrained(REPO, subfolder=<folder>, trust_remote_code=True)`;
`model.config.craft` says what it is (body, arm, source, the phase and step its weights ended on) and `model.config.activated` what its
body switches on. `crafts.json` at the top of the repo is the index; the README's tables are rendered from it. To package a subset,
set `CRAFTS` in C1 to a list and rerun C1 to C3.